In [2]:
import os
import glob
import zipfile
import pandas as pd
import numpy as np

# ==========================================
# [설정] 폴더 경로 및 파일 목록
# ==========================================
folder_path = '/Users/judetak/Desktop/산공카르텔경진대회/2025_생활인구 data'
output_file = '생활인구_LOCAL_RESD_202310_202510.csv'

# 250_LOCAL_RESD_202310.zip ~ 250_LOCAL_RESD_202510.zip 파일 탐색
zip_files = glob.glob(os.path.join(folder_path, '250_LOCAL_RESD_*.zip'))
zip_files.sort()

print(f'발견된 ZIP 파일 수: {len(zip_files)}')
for f in zip_files:
    print(f'  - {os.path.basename(f)}')

발견된 ZIP 파일 수: 25
  - 250_LOCAL_RESD_202310.zip
  - 250_LOCAL_RESD_202311.zip
  - 250_LOCAL_RESD_202312.zip
  - 250_LOCAL_RESD_202401.zip
  - 250_LOCAL_RESD_202402.zip
  - 250_LOCAL_RESD_202403.zip
  - 250_LOCAL_RESD_202404.zip
  - 250_LOCAL_RESD_202405.zip
  - 250_LOCAL_RESD_202406.zip
  - 250_LOCAL_RESD_202407.zip
  - 250_LOCAL_RESD_202408.zip
  - 250_LOCAL_RESD_202409.zip
  - 250_LOCAL_RESD_202410.zip
  - 250_LOCAL_RESD_202411.zip
  - 250_LOCAL_RESD_202412.zip
  - 250_LOCAL_RESD_202501.zip
  - 250_LOCAL_RESD_202502.zip
  - 250_LOCAL_RESD_202503.zip
  - 250_LOCAL_RESD_202504.zip
  - 250_LOCAL_RESD_202505.zip
  - 250_LOCAL_RESD_202506.zip
  - 250_LOCAL_RESD_202507.zip
  - 250_LOCAL_RESD_202508.zip
  - 250_LOCAL_RESD_202509.zip
  - 250_LOCAL_RESD_202510.zip


In [ ]:
# ==========================================
# [설정] 컬럼명 정의 (원본 컬럼명 그대로 사용, rename 없음)
# ==========================================
COL_DATE = '일자'
COL_TIME = '시간'
COL_GRID = '250M격자'
COL_POP  = '생활인구합계'

TARGET_COLS = [COL_DATE, COL_TIME, COL_GRID, COL_POP]

def read_csv_from_zip(z, csv_file):
    encodings = ['utf-8', 'utf-8-sig', 'cp949']
    for enc in encodings:
        try:
            with z.open(csv_file) as f:
                df = pd.read_csv(f, encoding=enc)
                # 실제 컬럼 중 TARGET_COLS에 해당하는 것만 추출
                matched = [c for c in df.columns if any(t in c for t in TARGET_COLS)]
                # 정확히 일치하는 컬럼이 있으면 우선 사용
                exact = [c for c in df.columns if c in TARGET_COLS]
                use_cols = exact if exact else matched
                return df[use_cols]
        except (UnicodeDecodeError, Exception):
            continue
    raise ValueError(f'인코딩 실패: {csv_file}')

print('설정 완료')

설정 완료


In [ ]:
# ==========================================
# STEP 1: 파일 읽기 및 결합
# ==========================================
df_list = []
print('📦 ZIP 파일에서 필요한 데이터만 추출합니다...')

for zip_file in zip_files:
    month_name = os.path.basename(zip_file)
    print(f'\n  처리 중: {month_name}')

    with zipfile.ZipFile(zip_file, 'r') as z:
        csv_files = [
            f for f in z.namelist()
            if f.endswith('.csv')
            and '__MACOSX' not in f
            and not f.split('/')[-1].startswith('._')
        ]
        print(f'    └─ 내부 CSV 파일 수: {len(csv_files)}')

        for csv_file in csv_files:
            try:
                df = read_csv_from_zip(z, csv_file)
                # 컬럼명 정확히 맞추기
                df.columns = [c.strip() for c in df.columns]
                if COL_POP in df.columns:
                    df[COL_POP] = pd.to_numeric(df[COL_POP], errors='coerce')
                df_list.append(df)
                print(f'      ✓ {csv_file.split("/")[-1]} ({len(df):,}행) | 컬럼: {list(df.columns)}')
            except Exception as e:
                print(f'      ✗ 오류 ({csv_file}): {e}')

print(f'\n🔄 전체 데이터 병합 중... (총 {len(df_list)}개 파일)')
full_df = pd.concat(df_list, ignore_index=True)
print(f'✅ 병합 완료: {full_df.shape[0]:,}행 × {full_df.shape[1]}열')
print(f'\n컬럼 목록: {list(full_df.columns)}')
full_df.head()

📦 ZIP 파일에서 필요한 데이터만 추출합니다...

  처리 중: 250_LOCAL_RESD_202310.zip
    └─ 내부 CSV 파일 수: 31
      ✓ 250_LOCAL_RESD_20231001.csv (253,483행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231002.csv (254,195행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231003.csv (254,056행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231004.csv (254,097행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231005.csv (253,969행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231006.csv (254,037행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231007.csv (254,225행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231008.csv (254,117행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231009.csv (254,364행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231010.csv (254,137행) | 컬럼: ['일자', '시간', '250M격자', '생활인구합계']
      ✓ 250_LOCAL_RESD_20231011.csv (253,981행) | 컬럼: 

,일자,시간,250M격자,생활인구합계
0,20231001,0,다사52255350,6.50
1,20231001,0,다사52255375,14.14
2,20231001,0,다사52505325,20.76
3,20231001,0,다사52505350,955.40
4,20231001,0,다사52505375,59.34


In [ ]:
full_df.to_csv('생활인구_LOCAL_RESD_202310_202510_raw .csv')

In [4]:

full_df_copy = pd.read_csv('생활인구_LOCAL_RESD_202310_202510_raw .csv')

COL_DATE = '일자'
COL_TIME = '시간'
COL_GRID = '250M격자'
COL_POP  = '생활인구합계'

original_pop = full_df_copy[COL_POP].copy()


print('=== 기본 정보 ===')
print(f'총 행수: {len(full_df_copy):,}')
print(f'결측치 수: {full_df_copy[COL_POP].isna().sum():,}')
print(f'\n기간 범위:')
print(f'  최솟값: {full_df_copy[COL_DATE].min()}')
print(f'  최댓값: {full_df_copy[COL_DATE].max()}')
print(f'\n총생활인구수 기초통계:')
print(full_df_copy[COL_POP].describe())

=== 기본 정보 ===
총 행수: 194,035,039
결측치 수: 10,060,698

기간 범위:
  최솟값: 20231001
  최댓값: 20251031

총생활인구수 기초통계:
count    1.839743e+08
mean     1.076069e+03
std      1.471755e+03
min      4.000000e+00
25%      4.552000e+01
50%      3.891700e+02
75%      1.716820e+03
max      7.127804e+04
Name: 생활인구합계, dtype: float64


In [5]:

print('🔍 격자별 이상치 제거 + EWMA 보간 중...')

results = []
# 전체 탐색 대신 groupby로 미리 격자별로 쪼개놓고 루프를 돕니다. (속도 수십 배 향상)
grouped = full_df_copy.groupby(COL_GRID)
total = len(grouped)

for i, (grid, chunk) in enumerate(grouped):
    # 진행 상황을 더 자주 확인하려면 50이나 100 단위로 줄이세요
    if i % 100 == 0:
        print(f'  진행: {i:,} / {total:,} 격자 처리 중...')

    chunk = chunk.copy()
    chunk = chunk.sort_values(by=[COL_DATE, COL_TIME])
    orig = chunk[COL_POP].copy()

    # 1. 시간대별 이상치 제거 (내부 for문 대신 transform 사용)
    time_grouped = chunk.groupby(COL_TIME)[COL_POP]
    
    # 각 시간대별 하위 10%, 상위 10%를 벡터 연산으로 한 번에 계산
    p10 = time_grouped.transform(lambda x: x.quantile(0.10))
    p90 = time_grouped.transform(lambda x: x.quantile(0.90))
    p_range = p90 - p10
    
    lo = p10 - 3 * p_range
    hi = p90 + 3 * p_range
    
    outlier = (chunk[COL_POP] < lo) | (chunk[COL_POP] > hi)
    chunk.loc[outlier, COL_POP] = np.nan

    # 2. EWMA + bfill 보간
    chunk[COL_POP] = chunk[COL_POP].fillna(
        chunk[COL_POP].ewm(span=3, min_periods=1, ignore_na=True).mean()
    ).bfill()

    # 3. 전체 기간 NaN이면 원본 복원
    if chunk[COL_POP].isna().all():
        chunk[COL_POP] = orig

    results.append(chunk)

print('🔄 결과 병합 중...')
full_df_copy = pd.concat(results, ignore_index=True)
print(f'✅ 완료! 잔여 결측치: {full_df_copy[COL_POP].isna().sum():,}')

🔍 격자별 이상치 제거 + EWMA 보간 중...
  진행: 0 / 8,886 격자 처리 중...
  진행: 100 / 8,886 격자 처리 중...
  진행: 200 / 8,886 격자 처리 중...
  진행: 300 / 8,886 격자 처리 중...
  진행: 400 / 8,886 격자 처리 중...
  진행: 500 / 8,886 격자 처리 중...
  진행: 600 / 8,886 격자 처리 중...
  진행: 700 / 8,886 격자 처리 중...
  진행: 800 / 8,886 격자 처리 중...
  진행: 900 / 8,886 격자 처리 중...
  진행: 1,000 / 8,886 격자 처리 중...
  진행: 1,100 / 8,886 격자 처리 중...
  진행: 1,200 / 8,886 격자 처리 중...
  진행: 1,300 / 8,886 격자 처리 중...
  진행: 1,400 / 8,886 격자 처리 중...
  진행: 1,500 / 8,886 격자 처리 중...
  진행: 1,600 / 8,886 격자 처리 중...
  진행: 1,700 / 8,886 격자 처리 중...
  진행: 1,800 / 8,886 격자 처리 중...
  진행: 1,900 / 8,886 격자 처리 중...
  진행: 2,000 / 8,886 격자 처리 중...
  진행: 2,100 / 8,886 격자 처리 중...
  진행: 2,200 / 8,886 격자 처리 중...
  진행: 2,300 / 8,886 격자 처리 중...
  진행: 2,400 / 8,886 격자 처리 중...
  진행: 2,500 / 8,886 격자 처리 중...
  진행: 2,600 / 8,886 격자 처리 중...
  진행: 2,700 / 8,886 격자 처리 중...
  진행: 2,800 / 8,886 격자 처리 중...
  진행: 2,900 / 8,886 격자 처리 중...
  진행: 3,000 / 8,886 격자 처리 중...
  진행: 3,100 / 8,886 격자 처리 중...
  

In [6]:
full_df_copy.head(10)

,Unnamed: 0,일자,시간,250M격자,생활인구합계
0,5864,20231001,0,다사35005075,4.12
1,16427,20231001,1,다사35005075,4.12
2,132587,20231001,2,다사35005075,4.12
3,185408,20231001,3,다사35005075,4.12
4,195972,20231001,4,다사35005075,4.12
5,206536,20231001,5,다사35005075,4.12
6,217103,20231001,6,다사35005075,4.65
7,227678,20231001,7,다사35005075,5.06
8,238237,20231001,8,다사35005075,5.30
9,248793,20231001,9,다사35005075,5.50


In [7]:
zero_count = (full_df_copy[COL_POP] == 0).sum()
print(f'총생활인구수가 0인 행: {zero_count:,}개')
full_df_copy[full_df_copy[COL_POP] == 0].head()

총생활인구수가 0인 행: 0개


,Unnamed: 0,일자,시간,250M격자,생활인구합계


In [8]:
na_count = full_df_copy[COL_POP].isna().sum()
print(f'잔여 결측치: {na_count:,}개')

잔여 결측치: 1,726,560개


In [9]:
# ==========================================
# [진단] 잔여 결측치 원인 분석
# ==========================================
# EWMA와 bfill로도 채워지지 않는 경우 = 해당 격자코드의 전체 기간이 모두 NaN
# (원본부터 결측이었거나, 이상치 제거로 전부 날아간 격자)

na_mask = full_df_copy[COL_POP].isna()

# 잔여 결측 격자 목록
na_grids = full_df_copy.loc[na_mask, COL_GRID].unique()
print(f'잔여 결측치가 있는 격자 수: {len(na_grids):,}개')
print(f'전체 격자 수: {full_df_copy[COL_GRID].nunique():,}개')
print(f'비율: {len(na_grids)/full_df_copy[COL_GRID].nunique()*100:.1f}%')

# 격자별 결측 행 수 확인
na_by_grid = (
    full_df_copy[na_mask]
    .groupby(COL_GRID)
    .size()
    .reset_index(name='결측행수')
    .sort_values('결측행수', ascending=False)
)
total_rows_per_grid = full_df_copy.groupby(COL_GRID).size().reset_index(name='전체행수')
na_summary = na_by_grid.merge(total_rows_per_grid, on=COL_GRID)
na_summary['결측비율(%)'] = (na_summary['결측행수'] / na_summary['전체행수'] * 100).round(1)

print('\n--- 결측 심각도 분포 ---')
print(f'전체 기간 100% 결측 격자: {(na_summary["결측비율(%)"] == 100).sum():,}개')
print(f'50% 이상 결측 격자:       {(na_summary["결측비율(%)"] >= 50).sum():,}개')
print(f'50% 미만 결측 격자:       {(na_summary["결측비율(%)"] < 50).sum():,}개')
print('\n상위 10개 격자:')
print(na_summary.head(10).to_string(index=False))

잔여 결측치가 있는 격자 수: 160개
전체 격자 수: 8,886개
비율: 1.8%

--- 결측 심각도 분포 ---
전체 기간 100% 결측 격자: 160개
50% 이상 결측 격자:       160개
50% 미만 결측 격자:       0개

상위 10개 격자:
    250M격자  결측행수  전체행수  결측비율(%)
다사60255575 36048 36048    100.0
다사55506000 34704 34704    100.0
다사54505500 18696 18696    100.0
다사37255300 18696 18696    100.0
다사35005100 18024 18024    100.0
다사54255775 18024 18024    100.0
다사39255475 18024 18024    100.0
다사51506050 18024 18024    100.0
다사56005900 18024 18024    100.0
다사56005825 18024 18024    100.0


In [12]:
# ==========================================
# STEP 4: 100% 결측 격자 → 원본값 복원 (인덱스 리셋 대응)
# ==========================================

# full_df_copy와 원본을 격자+일자+시간 기준으로 merge해서 복원
# (concat 후 ignore_index=True로 인덱스가 바뀌었기 때문에 직접 매핑)
original_df = pd.read_csv('생활인구_LOCAL_RESD_202310_202510_raw .csv')
original_df.columns = [c.strip() for c in original_df.columns]
original_df[COL_POP] = pd.to_numeric(original_df[COL_POP], errors='coerce')

# 100% 결측 격자 식별
na_ratio = full_df_copy.groupby(COL_GRID)[COL_POP].apply(lambda x: x.isna().all())
fully_na_grids = na_ratio[na_ratio].index
print(f'원본값으로 복원할 격자 수: {len(fully_na_grids):,}개')

# 해당 격자만 원본에서 가져와서 덮어쓰기
orig_subset = original_df[original_df[COL_GRID].isin(fully_na_grids)]
full_df_copy = full_df_copy[~full_df_copy[COL_GRID].isin(fully_na_grids)]
full_df_copy = pd.concat([full_df_copy, orig_subset], ignore_index=True)
full_df_copy = full_df_copy.sort_values(by=[COL_GRID, COL_DATE, COL_TIME])

print(f'최종 결측치 수: {full_df_copy[COL_POP].isna().sum():,}개')

원본값으로 복원할 격자 수: 0개
최종 결측치 수: 88,711개


In [13]:
# 원본 raw에서 결측치 확인
original_df = pd.read_csv('생활인구_LOCAL_RESD_202310_202510_raw .csv')
original_df.columns = [c.strip() for c in original_df.columns]
original_df[COL_POP] = pd.to_numeric(original_df[COL_POP], errors='coerce')

print(f'원본 결측치 수: {original_df[COL_POP].isna().sum():,}개')
print(f'현재 결측치 수: {full_df_copy[COL_POP].isna().sum():,}개')

# 결측치가 특정 격자에 몰려있는지, 아니면 분산되어 있는지 확인
na_mask = full_df_copy[COL_POP].isna()
na_by_grid = full_df_copy[na_mask].groupby(COL_GRID).size().sort_values(ascending=False)
print(f'\n결측치 있는 격자 수: {len(na_by_grid):,}개')
print(f'격자당 평균 결측행: {na_by_grid.mean():.1f}개')
print(f'\n상위 10개:')
print(na_by_grid.head(10))

원본 결측치 수: 10,060,698개
현재 결측치 수: 88,711개

결측치 있는 격자 수: 160개
격자당 평균 결측행: 554.4개

상위 10개:
250M격자
다사60255575    1828
다사55506000    1695
다사59504850    1177
다사64005925    1172
다사50253975    1147
다사50755350    1138
다사61754100    1132
다사65255400    1110
다사56005825    1096
다사50754000    1094
dtype: int64


In [14]:
# 잔여 결측치: 같은 격자 + 같은 시간대의 중앙값으로 채우기
grid_time_median = full_df_copy.groupby([COL_GRID, COL_TIME])[COL_POP].transform('median')
full_df_copy[COL_POP] = full_df_copy[COL_POP].fillna(grid_time_median)

print(f'격자+시간대 중앙값 후 잔여 결측치: {full_df_copy[COL_POP].isna().sum():,}개')

격자+시간대 중앙값 후 잔여 결측치: 4개


In [15]:
full_df_copy[COL_POP] = full_df_copy[COL_POP].fillna(0)
print(f'최종 결측치: {full_df_copy[COL_POP].isna().sum():,}개')

최종 결측치: 0개


In [16]:
# ==========================================
# STEP 4: 저장
# ==========================================
save_path = os.path.join(folder_path, output_file)
full_df_copy.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f'✅ 저장 완료: {save_path}')
print(f'   파일 크기: {os.path.getsize(save_path) / 1024 / 1024:.1f} MB')

✅ 저장 완료: /Users/judetak/Desktop/산공카르텔경진대회/2025_생활인구 data/생활인구_LOCAL_RESD_202310_202510.csv
   파일 크기: 8149.0 MB


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/Users/judetak/Desktop/산공카르텔경진대회/2025_생활인구 data/생활인구_LOCAL_RESD_202310_202510.csv')
df_grid = pd.read_csv('/Users/judetak/Desktop/산공카르텔경진대회/격자_250m_4326.csv')

In [3]:
df.tail()

,Unnamed: 0,일자,시간,250M격자,생활인구합계
194035034,182643512,20250917,2,다사72005025,0.00
194035035,182653949,20250917,3,다사72005025,864.34
194035036,182664385,20250917,4,다사72005025,0.00
194035037,183748084,20250921,10,다사72005025,24.31
194035038,184269620,20250923,11,다사72005025,12.18
